<a href="https://colab.research.google.com/github/ArthurrCr/cloudband/blob/main/notebooks/00_baselines/score_ocm_pixbox_s2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get -qq install aria2 > /dev/null 2>&1
!pip install --quiet omnicloudmask==1.7.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 9.5 MB/s eta 0:00:00


In [2]:
REPO_URL = "https://github.com/ArthurrCr/cloudband.git"
PROJECT_DIR = "/content/cloudband"
BRANCH = "main"

import os
import sys

if not os.path.exists(PROJECT_DIR):
    !git clone --quiet {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git fetch --quiet origin && git reset --quiet --hard origin/{BRANCH}

SRC_DIR = f"{PROJECT_DIR}/src"
os.chdir(PROJECT_DIR)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

!PYTHONPATH={SRC_DIR} python -m pytest tests -q

........................................................................ [ 57%]
.....................................................                    [100%]
=============================== warnings summary ===============================
tests/contract/test_pipeline.py::test_wrong_raster_size_is_rejected
tests/contract/test_pipeline.py::test_duplicate_predictions_are_rejected
tests/contract/test_pipeline.py::test_attach_and_score_round_trip
tests/contract/test_pipeline.py::test_pooled_counts_equal_whole_collection
  /usr/local/lib/python3.13/dist-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
    dataset = writer(

tests/contract/test_pipeline.py::test_wrong_raster_size_is_rejected
tests/contract/test_pipeline.py::test_attach_and_score_round_trip
tests/contract/test_pipeline.py::test_pooled_counts_equal_whole_collection
  /usr/local/lib/python3.13/dist-packages/rasterio/__init__.py:367: No

In [3]:
from pathlib import Path

import pandas as pd

from cloudband.acquisition import manifest, zenodo
from cloudband.baselines import ocm
from cloudband.colab.session import reload_package, start
from cloudband.labels import pixbox
from cloudband.pipelines import pixbox_s2
from cloudband.eval.report import as_percentages, to_frame

reload_package("cloudband")

In [4]:
def report(position, total):
    print(f"{position}/{total}", flush=True)

In [5]:
from google.colab import drive

drive.mount("/content/drive")

DATA_DIR = Path("/content/data/pixbox")
SCENES_DIR = Path("/content/data/scenes")
PRED_DIR = Path("/content/data/predictions")
RESULTS_DIR = Path("/content/drive/MyDrive/cloudband/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

session = start(PROJECT_DIR, require_accelerator=True)

Mounted at /content/drive
project: /content/cloudband
device: cuda:Tesla T4
free disk: 65.4 GiB
omnicloudmask: 1.7.0
rasterio: 1.5.1
pandas: 2.2.3
numpy: 2.1.3


In [6]:
zenodo.download(zenodo.LABELS, DATA_DIR)
!unzip -o -q {DATA_DIR}/{zenodo.LABELS_ARCHIVE} -d {DATA_DIR}

table = pixbox_s2.load_reference(DATA_DIR / zenodo.LABELS_CSV)
print("scorable pixels:", len(table))
print("dropped:", pixbox.unavailable_pixel_counts(pd.read_csv(DATA_DIR / zenodo.LABELS_CSV)))

scorable pixels: 16801
dropped: {'total': 550, 'clear': 149, 'cloud': 346, 'shadow': 82}


In [7]:
archive = zenodo.download(zenodo.SCENES, Path("/content/data"))
extracted = zenodo.extract_scenes(archive, SCENES_DIR)
print("extracted:", len(extracted))

extracted: 28


In [8]:
expected = [
    name.replace(".SAFE", "")
    for pid, name in pixbox.PRODUCT_ID_TO_SCENE.items()
    if pid not in pixbox.UNAVAILABLE_PRODUCT_IDS
]
statuses = manifest.check_collection(SCENES_DIR, expected)
print(manifest.summarise(statuses))
manifest.require_complete(statuses)

{'expected': 28, 'usable': 28, 'missing': [], 'total_gib': 20.18}


In [9]:
config = ocm.InferenceConfig(model_version=ocm.LATEST_MODEL_VERSION)
scene_paths = [status.path for status in statuses]

masks = ocm.predict_scenes(scene_paths, PRED_DIR, config)
print("masks:", len(masks))

PM_model_OCM_7.97_R_G_NIR_3_smp_regnety_(…): reconstructing file:   0%|          |  0.00B / 27.1MB            

PM_model_OCM_7.97_R_G_NIR_3_smp_regnety_(…): downloading bytes:           |  0.00B            

PM_model_OCM_7.97_R_G_NIR_3_smp_edgenext(…): reconstructing file:   0%|          |  0.00B / 30.7MB            

PM_model_OCM_7.97_R_G_NIR_3_smp_edgenext(…): downloading bytes:           |  0.00B            

Running inference using cuda float32:   0%|          | 0/28 [00:00<?, ?it/s]

masks: 28


In [10]:
scored = pixbox_s2.attach_predictions(table, PRED_DIR)
confusions = pixbox_s2.score(scored)
as_percentages(to_frame(confusions))

,tp,tn,fp,fn,ua,pa,oa,boa,f1,iou
experiment,,,,,,,,,,
clear,7690,7815,838,458,90.17,94.38,92.29,92.35,92.23,85.58
cloud,6899,8530,448,924,93.90,88.19,91.83,91.60,90.96,83.41
shadow,759,15470,167,405,81.97,65.21,96.60,82.07,72.63,57.02
